# 40 · Capstone: design the Llama 70B serving platform
Level L6 · Part 6 · core · ~35 min · builds on 25, 26, 28, 34, 35

Time: about 35 min. No key and no GPU. Every number is arithmetic from the model, the H200 spec sheet and the inputs you set in step 1. Step 7 starts a **SIMULATED** two-region lab in Docker (mock engines, no model runs). Without Docker, step 7 shows the **recorded output** (`data/recorded/40.json`), labelled as such.

Machine: any laptop with the topic's Python. Docker is optional. Ports: 8400 (global gateway of the lab).

## Goal
By the end you have the numbers for a 6-to-10-page design of the Brightside serving platform: 10,000 requests per second at peak, two regions, three tenants. Every number in the design traces to one input you set, one earlier lesson, or one measurement of yours.

## Why it matters
- **Without a capstone that uses numbers:** a design is a list of parts ("vLLM on a cluster, with autoscaling and a gateway"). Nobody can say how many GPUs it needs, what it costs per million tokens, how many requests fail when a region goes down, or what changes at 10x traffic.
- **With it:** each design decision cites one number from this notebook. Change one input (the hit rate, the answer length, the price) and the fleet, the bill and the failure cost change with it. A reviewer can check the arithmetic.
- **When you don't need it:** a prototype for a few hundred users in one region. Lessons 25 and 26 are enough there. The full design is for a platform someone must pay for and keep running.

## The idea
The capstone joins five checkpoints. Lesson 25 picks the GPU, the precision and the tensor-parallel degree for a latency target. Lesson 26 turns a request rate into requests in flight, then into replicas, at a safe utilisation. Lesson 28 turns the fleet into cost per million tokens. Lesson 34 says when to split prefill and decode into two pools. Lesson 35 says how a new version reaches users, behind an eval gate. This notebook adds what a platform must also do: run in two regions, serve three tenants, survive a failed region and say what it costs.

An architect does the same before drawing a bridge. The load, the span and the safety factor are written on the drawing first. Each part of the drawing answers to one of those numbers.

```mermaid
flowchart TB
  C["clients: 3 tenants, 10,000 req/s at peak"] --> G["global gateway: region a first, b as backup"]
  G --> RA["region a gateway: least_conn"]
  G -. "region a down: retry" .-> RB["region b gateway"]
  RA --> A1["replica a1: H200, FP8, TP=1"]
  RA --> A2["replica a2"]
  RB --> B1["replica b1"]
  RB --> B2["replica b2"]
  A1 & A2 & B1 & B2 --> M["metrics: queue, KV use, TTFT, errors, cost per million tokens"]
```

## Requirements
Each row needs a number and its source.

| # | Decision | The number the design states | Built in step |
| --- | --- | --- | --- |
| 1 | Latency | TTFT p50 at most 0.5 s, 1,600-token prompt (lesson 25) | 1, 2 |
| 2 | Engine | H200, FP8, TP=1 (lesson 25), pinned image (lessons 08 to 27) | 2 |
| 3 | Replicas | per region: 30% headroom, at least 2 (lessons 25, 26) | 3 |
| 4 | Cache and routing | prefix-cache hit rate, sticky tenant routing (lessons 14, 30) | 2, 4 |
| 5 | Two regions | per-region sizing, degraded mode after a region loss (lesson 33) | 3, 7 |
| 6 | Failure | errors per failed region, and RPO (how much recent data a failure may lose) and RTO (how long the service may be down) per failure domain | 7, 8 |
| 7 | Tenants | share, rate ceiling (the most requests per second a tenant may send) and tier for each of the 3 tenants (lesson 36) | 6 |
| 8 | Cost | $ per GPU-hour (your quote), fleet cost per month, $ per million output tokens (lesson 28) | 5 |
| 9 | Disaggregation | prefill and decode split or not, and the test (lesson 34) | 8 |
| 10 | Rollout | golden set, shadow, canary steps, rollback rule (lesson 35) | 8 |
| 11 | What you cut | the lever, its saving, its quality or risk cost | 4 |

**Named only, not designed here:** Kubernetes objects such as LeaderWorkerSet and Kueue (read their own API docs before you name a field), engine TLS flags (read `vllm serve --help=all` for your version), vendor price lists (use your own quote).

## Build it
### Step 1: your inputs, in one place
Each value is an input of the design. The comment says where it comes from. Items marked ASSUMPTION are planning values to replace with your own measurement. The request shape is lesson 25's (1,600 prompt tokens, 200 answer tokens). The shared policy prefix of 1,500 tokens is the course's running example: every request starts with the same policy text.

In [1]:
import math, json, statistics, subprocess, time
from pathlib import Path

MINE = dict(
    peak_qps=10_000,                  # the desk's peak, all regions together (the course target)
    prompt=1_600, answer=200,         # a Brightside request (lesson 25); the answer is a planning ceiling, see below
    shared_prefix=1_500,              # the policy text every request starts with (the running example)
    hit=0.45,                         # ASSUMPTION: share of prompt tokens served from the prefix cache (Meta's 40-50%, RESEARCH.md §4)
    ttft_slo=0.5,                     # p50 time to first token, seconds (lesson 25)
    headroom=1.3,                     # 30% spare above peak (lesson 25)
    min_replicas=2,                   # per region, so one replica can fail (lesson 25)
    price_gpu_h=2.00,                 # ASSUMPTION: DeepSeek's own cost model, $ per GPU-hour (RESEARCH.md §8.6); enter your quote
    measured_rps=None,                # your own measurement of requests per second per replica (lesson 27); None uses the model
    tenants={"support desk": 0.6, "billing helper": 0.3, "internal evals": 0.1},   # ASSUMPTION: traffic split
)
assert MINE["hit"] <= MINE["shared_prefix"] / MINE["prompt"], "the cache cannot hit more than the shared prefix"
assert abs(sum(MINE["tenants"].values()) - 1) < 1e-9, "tenant shares must add up to 100%"
print({k: v for k, v in MINE.items() if k != "tenants"})

{'peak_qps': 10000, 'prompt': 1600, 'answer': 200, 'shared_prefix': 1500, 'hit': 0.45, 'ttft_slo': 0.5, 'headroom': 1.3, 'min_replicas': 2, 'price_gpu_h': 2.0, 'measured_rps': None}


The answer length of 200 is a ceiling for planning. Check it against the lesson 07 trace (made-up support traffic from a fixed-seed script, so it shows the method, not a real log). Here is the first row and the means.

In [2]:
rows = [json.loads(line) for line in Path("../data/07/support_trace.jsonl").read_text().splitlines() if line.strip()]
print("first row:", rows[0])
mean_prompt = statistics.mean(r["prompt_tokens"] for r in rows)
mean_answer = statistics.mean(r["output_tokens"] for r in rows)
print(f"{len(rows)} requests: mean prompt {mean_prompt:,.0f}, mean answer {mean_answer:,.0f} tokens")

first row: {'prompt_tokens': 1560, 'output_tokens': 24}
30 requests: mean prompt 1,553, mean answer 69 tokens


### Step 2: one replica, from lessons 24 and 25
The H200 numbers come from lesson 24's spec table (dense FP8 maths, memory, bandwidth). The model numbers are lesson 19's. The function below is lesson 25's arithmetic for one option (H200, FP8 weights, TP=1), with one new input: the prefix-cache hit rate, which skips that share of the prefill work. A hit cannot be above the shared prefix's share of the prompt, and the function refuses one that is.

In [3]:
GIB = 2**30
UTIL = 0.92                      # vLLM gpu_memory_utilization default (lesson 19)
P = 70_553_706_496               # Llama 3.1 70B parameters (lesson 19)
KV = 327_680                     # FP16 KV cache per token, 320 KiB, all GPUs together (lesson 03)
H200 = dict(mem=141e9, bw=4800e9, fp8=1979e12)   # lesson 24 spec table: memory, bytes per second, FP8 FLOP/s

def replica_rps(hit, answer=MINE["answer"]):
    """Requests per second, prefill time and decode time for one H200 replica (lesson 25's arithmetic)."""
    if not 0 <= hit <= MINE["shared_prefix"] / MINE["prompt"]:
        raise ValueError(f"hit {hit:.2f} is above the shared prefix share {MINE['shared_prefix'] / MINE['prompt']:.2f}")
    ctx = MINE["prompt"] + answer
    room = H200["mem"] * UTIL - P                # bytes left for KV after FP8 weights (1 byte each)
    batch = int(room / KV // ctx)                # requests the cache holds at once
    prefill = 2 * P * MINE["prompt"] / H200["fp8"] * (1 - hit)   # the cache skips the hit share
    step = (P + batch * ctx * KV) / H200["bw"]   # one decode step reads the weights and every KV cache
    decode = answer * step / batch               # the batch shares each step
    return 1 / (prefill + decode), batch, prefill, decode

rps_model, batch, prefill, decode = replica_rps(MINE["hit"])
RPS = MINE["measured_rps"] or rps_model
print(f"batch {batch} requests; prefill {prefill:.3f} s (TTFT target {MINE['ttft_slo']} s); decode {decode:.3f} s")
print(f"requests per second per replica: {RPS:.2f} ({'your measurement' if MINE['measured_rps'] else 'model, not measured: replace with your lesson 27 number'})")
print(f"no prefix cache: {replica_rps(0)[0]:.2f} req/s (lesson 25 printed 5.95)")
assert prefill <= MINE["ttft_slo"], "the prefill alone misses the TTFT target"

batch 100 requests; prefill 0.063 s (TTFT target 0.5 s); decode 0.054 s
requests per second per replica: 8.57 (model, not measured: replace with your lesson 27 number)
no prefix cache: 5.95 req/s (lesson 25 printed 5.95)


**Common mistake: a hit rate with no shared-prefix check.** The cell below fails on purpose.

In [4]:
replica_rps(0.97)   # Common mistake: a hit rate taken from a lab row, with no shared-prefix check. This cell fails on purpose.

ValueError: hit 0.97 is above the shared prefix share 0.94

**What you should see:** the error. A 97% hit is not possible here: only 1,500 of the 1,600 prompt tokens are shared, so at most 94% can come from the cache. The lab's 97% (lesson 18) came from one repeated prompt. A design must use the hit rate of its own traffic.

### Step 3: the fleet, in two regional options
Two rules decide the size. The first is lesson 26's: replicas = requests per second × headroom ÷ requests per second per replica, with at least two per region. The second is the two-region rule: if one region fails, can the other one carry the load? **Option A** sizes each region for the full peak, so a region loss keeps 10,000 req/s. **Option B** sizes each region for half the peak and accepts a degraded mode when a region is lost: the service keeps running at lower capacity and refuses the excess early.

In [5]:
def fleet(qps, rps=RPS):
    """Replicas in one region for a request rate, with headroom and a floor of min_replicas."""
    return max(MINE["min_replicas"], math.ceil(qps * MINE["headroom"] / rps))

def monthly(gpus):
    return gpus * MINE["price_gpu_h"] * 730          # 730 hours in a month (lesson 28)

per_a = fleet(MINE["peak_qps"]); gpus_a = 2 * per_a
per_b = fleet(MINE["peak_qps"] / 2); gpus_b = 2 * per_b
served_b = min(1.0, per_b * RPS / MINE["peak_qps"])  # share of peak one region can serve alone (no headroom spent)
print(f"{'option':<36}{'per region':>11}{'GPUs':>7}{'$ per month':>14}")
print(f"{'A: each region at full peak':<36}{per_a:>11,}{gpus_a:>7,}{monthly(gpus_a):>14,.0f}")
print(f"{'B: each region at half, degraded':<36}{per_b:>11,}{gpus_b:>7,}{monthly(gpus_b):>14,.0f}")
print(f"option B after a region loss: the survivor serves {served_b:.0%} of the peak, the rest is refused early (lesson 33)")

option                               per region   GPUs   $ per month
A: each region at full peak               1,518  3,036     4,432,560
B: each region at half, degraded            759  1,518     2,216,280
option B after a region loss: the survivor serves 65% of the peak, the rest is refused early (lesson 33)


**Read it.** Option B costs half as much and serves about two thirds of the peak after a region loss. A desk that promises 10,000 requests a second through a region loss needs option A. The design must say which promise it makes; the table is the price of each one.

Option B's "about two thirds" is the survivor's capacity with no headroom spent: 759 replicas at 8.57 req/s each, against the 10,000 peak.

### Step 4: what gets cut, and what each cut costs
Two levers change the fleet on option A: the prefix-cache hit rate (lesson 14, kept warm by the routing of lesson 30) and the answer length (the lesson 07 trace mean is far below the 200 ceiling). The table prices each lever alone, then both together. The 69-token mean is only a guide: size the design on a measured high percentile of answer lengths, not the mean. The Cloudflare 80% hit ratio (RESEARCH.md §8.6) is a company's figure with session affinity, not a result for this desk; it is a target to test.

In [6]:
scenarios = [
    ("baseline (option A)", MINE["hit"], MINE["answer"]),
    ("hit 80% with session affinity", 0.80, MINE["answer"]),
    (f"answer {mean_answer:.0f} tokens (trace mean)", MINE["hit"], round(mean_answer)),
    ("hit 80% and the trace answer length", 0.80, round(mean_answer)),
]
print(f"{'scenario':<38}{'req/s/replica':>14}{'GPUs':>7}{'$ per month':>14}")
for name, hit, ans in scenarios:
    rps_s = replica_rps(hit, ans)[0]
    gpus = 2 * fleet(MINE["peak_qps"], rps_s)
    print(f"{name:<38}{rps_s:>14.2f}{gpus:>7,}{monthly(gpus):>14,.0f}")

scenario                               req/s/replica   GPUs   $ per month
baseline (option A)                             8.57  3,036     4,432,560
hit 80% with session affinity                  13.02  1,998     2,917,080
answer 69 tokens (trace mean)                  12.50  2,080     3,036,800
hit 80% and the trace answer length            24.96  1,042     1,521,320


**Read it.** The cache is the bigger single lever: 80% hits cut the fleet by about a third (3,036 to 1,998 GPUs). The shorter answers cut it by about the same share alone. Together they cut it by two thirds (to 1,042 GPUs, about $1.5 million a month). Neither is free. Shorter answers need a quality check against the golden set (lesson 35). A cache hit rate needs the server's own counters to prove it (lesson 28). Until then the design uses the planning values of step 1, and each cut is written down with its risk.

### Step 5: cost per million output tokens
Lesson 28's rule: cost per served token is the hourly price over the tokens served in that hour. The fleet at peak serves `peak × answer` output tokens a second. The number depends on the utilisation, so the next line prints it. Headroom 1.3 means the replicas run at 1 ÷ 1.3 of their planned capacity at peak. Lesson 26 targets 70%, which would need a headroom of 1 ÷ 0.7.

In [7]:
gpus = gpus_a
cost_h = gpus * MINE["price_gpu_h"]
out_per_h = MINE["peak_qps"] * MINE["answer"] * 3600
print(f"fleet: {gpus:,} GPUs at ${MINE['price_gpu_h']:.2f} per hour = ${cost_h:,.0f} per hour")
print(f"at peak: {out_per_h / 3600 / 1e6:.2f} million output tokens per second, ${cost_h / out_per_h * 1e6:.2f} per million output tokens")
gpus_70 = 2 * math.ceil(MINE["peak_qps"] / 0.7 / RPS)     # both regions at full peak, planned at 70% (lesson 26)
print(f"utilisation at peak: {1 / MINE['headroom']:.0%} with headroom {MINE['headroom']}")
print(f"planned at 70% instead: {gpus_70:,} GPUs, ${monthly(gpus_70):,.0f} per month (option A)")

fleet: 3,036 GPUs at $2.00 per hour = $6,072 per hour
at peak: 2.00 million output tokens per second, $0.84 per million output tokens
utilisation at peak: 77% with headroom 1.3
planned at 70% instead: 3,336 GPUs, $4,870,560 per month (option A)


**Read it.** The $0.84 counts all 3,036 GPUs, the standby region's 1,518 included, because option A pays for both regions. Only half of the fleet serves at any time while the other region stands by. Serving from both regions all the time is a different design: each region then carries part of the traffic, and the price per token depends on how that load is split.

### Step 6: three tenants on one platform
Each tenant gets a share of the traffic, a rate ceiling (its share plus 20%, so one tenant's burst cannot take the others' capacity), and a tier. The top tier keeps its queue short and its requests first. The ceiling is a gateway rule (lesson 36: per-key limits and budgets); the platform cannot isolate tenants inside one engine process (vLLM's security guidance).

In [8]:
print(f"{'tenant':<18}{'share':>7}{'req/s at peak':>15}{'ceiling req/s':>15}{'tokens/min ceiling':>22}")
tier = {"support desk": "tier 1: first in queue", "billing helper": "tier 2", "internal evals": "tier 3: first to be refused"}
for name, share in MINE["tenants"].items():
    qps = MINE["peak_qps"] * share
    ceiling = qps * 1.2
    tpm = ceiling * 60 * (MINE["prompt"] + MINE["answer"])
    print(f"{name:<18}{share:>7.0%}{qps:>15,.0f}{ceiling:>15,.0f}{tpm:>22,.0f}   {tier[name]}")
print(f"sum of the peaks: {sum(MINE['peak_qps'] * s for s in MINE['tenants'].values()):,.0f} req/s = the platform peak")

tenant              share  req/s at peak  ceiling req/s    tokens/min ceiling
support desk          60%          6,000          7,200           777,600,000   tier 1: first in queue
billing helper        30%          3,000          3,600           388,800,000   tier 2
internal evals        10%          1,000          1,200           129,600,000   tier 3: first to be refused
sum of the peaks: 10,000 req/s = the platform peak


### Step 7: the two-region drill (SIMULATED, Docker)
The lab starts four replicas in two regions (a1, a2 and b1, b2), a gateway for each region and a global gateway on port 8400. The replicas are the lesson 33 mock: they run no model, they apply a queue limit and they answer with their name, so you can see where each request went. The global gateway prefers region a and retries on region b. The drill sends 10 requests in three phases: all healthy; region a stopped; region a started again.

To run the Docker cells, start Jupyter with `TUT_ENABLE=docker jupyter lab`. The Compose file ships with the course at `infra/compose/capstone-two-region.yml`; the notebook reads it as `../infra/compose/capstone-two-region.yml`.

The gateway settings in that file, in plain words:

| Setting | What it does in this lab |
| --- | --- |
| `least_conn` | sends each new request to the server with the fewest open connections |
| `max_fails=2`, `fail_timeout=10s` | after 2 failed attempts within 10 s, the gateway stops sending to that server for 10 s |
| connect timeout 1 s | a call to a stopped container gives up after 1 s, not after the 60 s default |
| sticky routing | not in this lab: a design rule (step 4) that keeps one tenant on one replica so its prefix cache stays warm |

In [9]:
import os, atexit
import httpx

COMPOSE = ["docker", "compose", "-p", "tut-kv-forge-15h-40", "-f", "../infra/compose/capstone-two-region.yml"]
GLOBAL = "http://127.0.0.1:8400"
BODY = {"model": "brightside-qwen-0.5b", "messages": [{"role": "user", "content": "Can I change my plan today?"}], "max_tokens": 8}
LIVE, RESULTS = False, {}

def compose(*args):
    r = subprocess.run([*COMPOSE, *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr[-600:])

`call_batch` sends ten calls through the global gateway and records where each one went. `wait_global` waits for the gateway's health check.

In [10]:
def call_batch(n=10):
    """n sequential requests through the global gateway: the replica name that answered, or the error."""
    out = []
    for _ in range(n):
        try:
            r = httpx.post(GLOBAL + "/v1/chat/completions", json=BODY, timeout=15)
            out.append(r.json()["replica"] if r.status_code == 200 else f"HTTP {r.status_code}")
        except httpx.HTTPError as e:
            out.append("ERR " + type(e).__name__)
    return out

def wait_global(limit=180):
    for _ in range(limit):
        try:
            if httpx.get(GLOBAL + "/health", timeout=2).status_code == 200:
                return True
        except httpx.TransportError:
            pass
        time.sleep(1)
    return False

print("lab file:", COMPOSE[-1], "| global gateway: 127.0.0.1:8400 | the replicas publish no ports")

lab file: ../infra/compose/capstone-two-region.yml | global gateway: 127.0.0.1:8400 | the replicas publish no ports


Start the lab. This cell needs Docker and runs only with `TUT_ENABLE=docker`. The first build takes a few minutes; later runs reuse the image.

In [11]:
compose("up", "-d", "--build")
atexit.register(lambda: LIVE and compose("down", "-v"))
LIVE = wait_global()
print("global gateway answers /health:", LIVE)

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Run the three phases. Each call records the replica that answered, or the error. Exact error counts vary from run to run. What to look for: a few early errors while the gateway still tries region a, then region b answers, with an occasional error; after region a returns and its wait passes, region a answers again.

In [12]:
if LIVE:
    RESULTS["healthy"] = call_batch()
    compose("stop", "a1", "a2")                    # region a goes down
    RESULTS["region a down"] = call_batch()
    compose("start", "a1", "a2")                   # region a comes back
    wait_global()
    time.sleep(11)                                 # longer than fail_timeout (10 s)
    RESULTS["region a back"] = call_batch()
    Path("../data/40").mkdir(parents=True, exist_ok=True)
    Path("../data/40/results.json").write_text(json.dumps(RESULTS, indent=1))
    print("live run saved to data/40/results.json (run-time file, not committed)")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Without Docker, the drill reads the recording (`data/recorded/40.json`), made on the course machine with the same Compose file. The table below reads whichever run exists, and says which one.

In [13]:
REC_PATH = Path("../data/recorded/40.json")
if not LIVE and not REC_PATH.exists():
    raise FileNotFoundError("no live run and no recording: start the lab with TUT_ENABLE=docker")
R = RESULTS if LIVE else json.loads(REC_PATH.read_text())["phases"]
print("source:", "this live run (SIMULATED engines)" if LIVE else "recorded output (SIMULATED engines, course machine)")
print(f"{'phase':<14}{'region a':>9}{'region b':>9}{'errors':>8}   the replicas that answered, in order")
for phase, answers in R.items():
    a = sum(x.startswith("a") for x in answers)
    b = sum(x.startswith("b") for x in answers)
    err = len(answers) - a - b
    print(f"{phase:<14}{a:>9}{b:>9}{err:>8}   {' '.join(answers)}")

source: recorded output (SIMULATED engines, course machine)
phase          region a region b  errors   the replicas that answered, in order
healthy              10        0       0   a1 a1 a1 a2 a1 a2 a1 a2 a1 a2
region a down         0        7       3   HTTP 504 HTTP 504 b1 b2 b1 b2 b1 b2 b1 HTTP 504
region a back        10        0       0   a1 a2 a1 a2 a1 a2 a1 a2 a1 a2


Stop the lab. Its volumes are removed, so the next run starts clean.

In [14]:
if LIVE:
    compose("down", "-v")
    LIVE = False
    print("lab stopped and its volumes removed")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**Read it.** Healthy: region a serves everything, alternating its two replicas. Region a down: the first calls that reach the dead region fail before the gateway marks it as failed, then region b serves the rest. The early errors are the cost of the detection window, and they depend on `max_fails` and the connect timeout. The last request is also an error, most likely because the gateway tried region a again once `fail_timeout` had passed (the drill does not time each call, so read that as a reading, not a measurement). Region a back: in this run, region a took all ten calls with no errors after an 11-second wait. That is one run after a wait. An earlier course-machine run without the wait sent calls to region b again before region a was used (RESEARCH.md §8.9), so the design does not promise a clean fail-back. It needs its own drill.

### Step 8: the failure table and the numbers block
The table states the recovery targets per failure domain. The values are design targets built from the lab and from the sources, not vendor promises. KV cache is never replicated: after an engine dies, the stream is lost and the request is sent again (re-prefill). Object storage for weights has its own replication: dual-region storage copies asynchronously, so the design must check a replica before it moves traffic (RESEARCH.md §8.8, Google Cloud Storage availability page, opened 2026-10-23).

| Failure domain | What is lost | RPO (data lost) | Recovery (RTO driver) |
| --- | --- | --- | --- |
| one engine or GPU | in-flight streams and that replica's KV cache | 0 for data; the streams are lost | health check, then re-prefill on another replica |
| one region | the region's fleet and its gateway | 0 for weights if replicated | global gateway fails over; option A keeps the peak |
| weights bucket | the model files | default dual-region RPO up to 12 h; turbo replication 15 min (premium) | publish, wait for the replica check, then shift traffic |
| gateway database | budgets and rate-limit counters | replication lag of the database | fail closed or open, decided per tenant tier |

Write the numbers block: the evidence page of the design. Each line is a number printed above.

In [15]:
numbers = [
    f"peak {MINE['peak_qps']:,} req/s; prompt {MINE['prompt']:,}, answer {MINE['answer']} tokens (ceiling; trace mean {mean_answer:.0f})",
    f"H200 FP8 TP=1: {RPS:.2f} req/s per replica ({'measured' if MINE['measured_rps'] else 'model'}); TTFT p50 {prefill:.3f} s (prefill only)",
    f"prefix hit {MINE['hit']:.0%} (assumption); headroom {MINE['headroom']}; minimum {MINE['min_replicas']} per region",
    f"option A: {per_a:,} per region, {gpus_a:,} GPUs, ${monthly(gpus_a):,.0f} per month at ${MINE['price_gpu_h']:.2f} per GPU-hour",
    f"option B: {per_b:,} per region, {gpus_b:,} GPUs; survivor serves {served_b:.0%} of peak",
    f"cost at peak (option A, both regions' {gpus:,} GPUs): ${cost_h / out_per_h * 1e6:.2f} per million output tokens",
    "tenants: " + ", ".join(f"{k} {v:.0%}" for k, v in MINE["tenants"].items()),
    "ITL p99 breach decides the split (lesson 34)",
    "golden set, shadow, 1/5/25% canary, rollback rule (lesson 35)",
    "drill: " + "; ".join(f"{p}: {sum(not x[0] in 'ab' for x in a)} errors of {len(a)}" for p, a in R.items()),
]
Path("../data/40").mkdir(parents=True, exist_ok=True)
Path("../data/40/design_numbers.md").write_text("# Numbers for the design (lesson 40)\n\n" + "\n".join(f"- {n}" for n in numbers) + "\n")
print("\n".join(f"- {n}" for n in numbers))
print("\nsaved to data/40/design_numbers.md (run-time file, not committed)")

- peak 10,000 req/s; prompt 1,600, answer 200 tokens (ceiling; trace mean 69)
- H200 FP8 TP=1: 8.57 req/s per replica (model); TTFT p50 0.063 s (prefill only)
- prefix hit 45% (assumption); headroom 1.3; minimum 2 per region
- option A: 1,518 per region, 3,036 GPUs, $4,432,560 per month at $2.00 per GPU-hour
- option B: 759 per region, 1,518 GPUs; survivor serves 65% of peak
- cost at peak (option A, both regions' 3,036 GPUs): $0.84 per million output tokens
- tenants: support desk 60%, billing helper 30%, internal evals 10%
- ITL p99 breach decides the split (lesson 34)
- golden set, shadow, 1/5/25% canary, rollback rule (lesson 35)
- drill: healthy: 0 errors of 10; region a down: 3 errors of 10; region a back: 0 errors of 10

saved to data/40/design_numbers.md (run-time file, not committed)


## Try it
Change `hit`, `price_gpu_h` and `tenants` in step 1 to your own values. Keep the hit rate at or below 0.94 for this prompt, and the tenant shares at 100%. Then re-run step 1 to step 8, so `design_numbers.md` reflects the new inputs. Write the one-paragraph decision: option A or B, the first lever to cut, and why. The page design is the `design_numbers.md` block from step 8 plus that decision.

In [16]:
# your code here
# Replace the empty string with your one-paragraph decision: option A or B, the first lever to cut, and why.
my_decision = ""
print(my_decision or "no decision written yet")

no decision written yet


## Reference solution
**Answer** — try it first, then expand the hidden cell below. One possible design, with the default inputs: option A, because the desk promises 10,000 requests a second through a region loss. The hit rate stays at the planning value until the counters measure it. The answer length stays at 200 until the quality check passes. The hidden cell prints the decision in the form the design needs, and a sample paragraph.

In [17]:
decision = dict(
    regions="option A: each region sized for the full peak",
    replicas_per_region=per_a,
    gpus=gpus_a,
    monthly_usd=round(monthly(gpus_a)),
    cut_first="prefix cache with session affinity (step 4), measured before it is counted",
    not_yet="disaggregated prefill: the 1,600-token prompt is medium, so split the pools only if the ITL p99 breaches its target (lesson 34)",
    rollout="golden set and gate, shadow, 1% then 5% then 25% canary, rollback on error or quality drop (lesson 35)",
)
for k, v in decision.items():
    print(f"{k:<20} {v}")
paragraph = (
    f"Option A: each region runs {per_a:,} replicas ({gpus_a:,} GPUs, ${monthly(gpus_a):,.0f} a month). "
    "The first lever to cut is the prefix cache with session affinity, measured on the server counters before it is counted. "
    "Disaggregation waits for an ITL p99 breach (lesson 34)."
)
print(paragraph)

regions              option A: each region sized for the full peak
replicas_per_region  1518
gpus                 3036
monthly_usd          4432560
cut_first            prefix cache with session affinity (step 4), measured before it is counted
not_yet              disaggregated prefill: the 1,600-token prompt is medium, so split the pools only if the ITL p99 breaches its target (lesson 34)
rollout              golden set and gate, shadow, 1% then 5% then 25% canary, rollback on error or quality drop (lesson 35)
Option A: each region runs 1,518 replicas (3,036 GPUs, $4,432,560 a month). The first lever to cut is the prefix cache with session affinity, measured on the server counters before it is counted. Disaggregation waits for an ITL p99 breach (lesson 34).


**Read it against published numbers.** The design has about 3,000 H200s for 10,000 requests a second with 200-token answers. That is a sizing claim, so check it against a published one. DeepSeek reported its own 24-hour system (February 2025): an average of 226.75 nodes of 8 H800 GPUs each, a cost of $87,072 a day at $2 per GPU-hour, and a 56.3% input-cache hit (RESEARCH.md §8.6). Its workload and its hardware differ from this desk's, so it is a sense check, not a target. A design that lands far from the published scale must say why. Cloudflare's 60% to 80% hit ratio with session affinity (2026-05) shows that the routing rule is a real lever, and Meta's 40% to 50% hit at 90% memory use (2025-09) is the planning value used here.


**What the arithmetic leaves out.** The model has no queueing: at 10,000 req/s, each replica's queue waits follow lesson 26's knee, which the headroom only partly covers. It assumes the H200 FP8 maths and bandwidth of the spec sheet, not a measured run; replace `measured_rps` with your lesson 27 number when you have a GPU. It assumes a flat price per GPU-hour, with no spot or reserved discount (lesson 28). Tenants are fixed shares, not a real traffic mix. The lab is SIMULATED: it shows the routing and the failure window, not the speed of a real engine.

## Key terms
- **Option A and option B** — the two regional sizing rules of step 3: full peak in each region, or half peak with a degraded mode after a region loss.
- **Degraded mode** — the service keeps running at lower capacity after a failure, refusing the excess early instead of queueing it.
- **RPO (recovery point objective)** — how much recent data a failure may lose, in time. **RTO (recovery time objective)** — how long the service may be down.
- **Headroom** — spare capacity above the peak, here 30% (lesson 25); it covers bursts and one replica that fails.
- **Tenant ceiling** (rate ceiling) — the most requests per second one tenant may send, its share plus 20% (lesson 36).
- **Simulated replica** — a mock that answers like an engine but runs no model (lesson 33); the lab uses it to show routing and failure, never answers.


**Interview:** "Design an LLM serving platform for 10,000 requests a second across two regions." Strong answer: state the TTFT target and the request shape. Pick the engine and the parallelism for one replica, then count replicas per region from requests per second with headroom and a floor of two. Decide whether a region loss must keep the full peak, because that doubles the fleet. Price it per million output tokens, name the cache and length levers, and say how the rollout is gated and what each failure costs.